# Complete model — CT-CLIP Temporal Difference Transformer (Colab)

Trains the temporal-progression model on the **cached CT-CLIP image embeddings** produced by
`ctclip_features_colab.ipynb` / `ctrate_download_colab.ipynb` (512-d per volume, in Drive).
The frozen CT-CLIP encoders
stay frozen; only a small difference/joint-attention module is trained.

**Design (matches the project thesis):**
- frozen CT-CLIP → two global 512-d vectors per pair (`v_prior`, `v_current`)
- trainable module: input proj + prior/current role embeddings + a learnable `e_diff` query +
  a tiny Transformer → a **difference embedding `d`** (back in CT-CLIP's 512-d joint space)
- **antisymmetry** flag: `d = g(c,p) − g(p,c)` so *stable* == the zero vector by construction
- **magnitude head** flag: scalar `||change||` to route *stable* (fixes cosine's blindness to no-change)
- each `(pair, finding)` is classified by cosine of the shared `d` against **that finding's**
  3 CT-CLIP text prototypes (worsened / stable / improved) → per-class AND per-disease F1
- optional **contrastive** aux loss: InfoNCE(`d`, the finding's `evidence` comparison sentence)

Uses the same CT-CLIP text tower (CXR-BERT + `to_text_latent`) to build prototypes, so text and
image live in one joint space. Runs on the cached tensors + a light text-encode pass; GPU optional.

**Strict CT-RATE domain split:** only pairs whose two volumes are `train_*` can contribute to
optimization. A patient-grouped tuning subset is carved from `train_*` for early stopping.
Pairs whose two volumes are `valid_*` are held out for one final test and never used for
training, early stopping, prototype construction, class weights, or model selection. The old
custom `subset_pairs.csv` train/val/test column is deliberately ignored.

## 1. Setup: clone CT-CLIP + our repo, install deps

In [ ]:
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -q -e transformer_maskgit
!pip install -q -e CT_CLIP
!pip install -q nibabel scipy huggingface_hub transformers scikit-learn tqdm
%cd /content
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path: sys.path.insert(0, p)
![ -d 3dCT ] || git clone https://github.com/nprakash1/3dCT.git 3dCT
!cd 3dCT && git pull -q
sys.path.append('/content/3dCT/scripts')
import ct_clip, transformer_maskgit; print('CT-CLIP import OK')

## 2. Mount Drive + config paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
csv.field_size_limit(10**9)

DRIVE      = '/content/drive/MyDrive/3dCT'
IMG_DIR    = f'{DRIVE}/ctclip_cache/img'          # 512-d per volume (from features notebook)
WEIGHTS    = f'{DRIVE}/ctclip_weights'            # CT-CLIP_v2.pt cached here
PROTO_PT   = f'{DRIVE}/ctclip_cache/proto_bank.pt'  # finding x class text prototypes (cached)
LAB        = '/content/3dCT/medgemma_labels_v3.jsonl'
LAB_DS     = '/content/3dCT/medgemma_labels (2).jsonl'  # dynamic_sentences/static_sentences

# Hub train is the ONLY development domain. Hub validation is final test only.
TUNE_FRAC  = 0.15   # patient-grouped early-stop subset drawn only from train_*
SPLIT_SEED = 2026   # stable SHA-256 partition; independent of Python hash/random state
REQUIRE_COMPLETE_HUB_VALID_FEATURES = True

CLASSES = ['worsened', 'stable', 'improved']      # == labels' `direction`
C2I = {c: i for i, c in enumerate(CLASSES)}
torch.manual_seed(0); np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| img cache exists:', os.path.isdir(IMG_DIR))

## 3. Load cached CT-CLIP IMAGE embeddings (512-d per volume)

In [ ]:
import glob
POOLED = {}
for fp in glob.glob(f'{IMG_DIR}/*.pt'):
    key = os.path.basename(fp)[:-3]                 # 'train_5200_a_1'
    POOLED[key] = torch.load(fp, map_location='cpu').float()   # (512,)
print('loaded image embeddings for', len(POOLED), 'volumes')
cache_domain = Counter('train' if k.startswith('train_') else
                       'hub_valid' if k.startswith('valid_') else 'unknown' for k in POOLED)
print('cache by CT-RATE Hub domain:', dict(cache_domain))
assert POOLED, 'No image embeddings found — run ctclip_features_colab.ipynb first.'
print('example dim:', tuple(next(iter(POOLED.values())).shape))

## 4. Build the frozen CT-CLIP text tower (for prototypes + evidence)
Reuses `CTCLIPEmbedder`; weights were cached to Drive by the features notebook.

In [ ]:
from huggingface_hub import login, hf_hub_download
from ctclip_utils import CTCLIPEmbedder, REPO_ID, CTCLIP_WEIGHTS_HF
login()  # paste READ token (only needed if weights not already on Drive)
os.makedirs(WEIGHTS, exist_ok=True)
wp = f'{WEIGHTS}/{os.path.basename(CTCLIP_WEIGHTS_HF)}'
if not os.path.exists(wp):
    wp = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset', local_dir=WEIGHTS)
emb = CTCLIPEmbedder(wp)
print('CT-CLIP text tower ready on', emb.device)

## 5. Load labels + enforce Hub train / Hub validation separation

`train_*` pairs are partitioned by **patient** into optimization `train` and early-stop `tune`.
`valid_*` pairs become final `test`. Cache availability can reduce usable examples but can never
change a pair's domain. Cross-domain/malformed pairs are rejected.

In [ ]:
import hashlib

def vkey(v): return v.replace('.nii.gz', '').replace('.nii', '')

def volume_domain(v):
    k = vkey(v).lower()
    if k.startswith('train_'): return 'hub_train'
    if k.startswith('valid_'): return 'hub_valid'
    return 'unknown'

def pair_domain(pv, cv):
    a, b = volume_domain(pv), volume_domain(cv)
    return a if a == b else 'cross_domain'

def dev_partition(patient):
    # Stable across machines/runs; never use Python's randomized hash().
    raw = f'{SPLIT_SEED}|{patient}'.encode('utf-8')
    u = int.from_bytes(hashlib.sha256(raw).digest()[:8], 'big') / 2**64
    return 'tune' if u < TUNE_FRAC else 'train'

recs = {}
for line in open(LAB, encoding='utf-8'):
    if not line.strip(): continue
    x = json.loads(line)
    recs[(x['patient'], x['prior_volume'], x['curr_volume'])] = x

# per-PAIR dynamic (comparison) sentences from the LLM static/dynamic split
dyn_of = {}
try:
    for line in open(LAB_DS, encoding='utf-8'):
        if not line.strip(): continue
        x = json.loads(line)
        ds = x.get('dynamic_sentences') or []
        if isinstance(ds, list): ds = ' '.join(s for s in ds if isinstance(s, str))
        dyn_of[(x['patient'], x['prior_volume'], x['curr_volume'])] = (ds or '').strip()
    print('loaded dynamic_sentences for', len(dyn_of), 'pairs')
except FileNotFoundError:
    print('WARN: dynamic-sentence file not found:', LAB_DS)

pair_ids = {}
examples = {'train': [], 'tune': [], 'test': []}
skipped = Counter()
candidate_pairs = Counter(); usable_pairs = Counter(); missing_hub_valid = []
for key, rec in recs.items():
    patient, pv, cv = key
    domain = pair_domain(pv, cv)
    if domain == 'hub_train':
        sp = dev_partition(patient)
    elif domain == 'hub_valid':
        sp = 'test'
    else:
        skipped[domain] += 1; continue
    if not rec.get('parse_ok'):
        skipped[f'no_label_{sp}'] += 1; continue
    candidate_pairs[sp] += 1
    if vkey(pv) not in POOLED or vkey(cv) not in POOLED:
        skipped[f'no_feature_{sp}'] += 1
        if sp == 'test': missing_hub_valid.append(key)
        continue
    usable_pairs[sp] += 1
    for fd in rec.get('findings', []):
        if fd.get('tier') != 'explicit':
            skipped['not_explicit'] += 1; continue
        d = fd.get('direction'); f = fd.get('finding')
        if d not in C2I or not f:
            skipped['bad_dir'] += 1; continue
        pid = pair_ids.setdefault(key, len(pair_ids))
        examples[sp].append({'vp': vkey(pv), 'vc': vkey(cv), 'patient': patient,
                             'finding': f, 'hub_domain': domain,
                             'y': C2I[d], 'pid': pid,
                             'evidence': fd.get('evidence', '') or '',
                             'dynamic': dyn_of.get(key, '')})

# Non-negotiable leakage guards.
for sp in ['train', 'tune']:
    assert all(e['hub_domain'] == 'hub_train' and e['vp'].startswith('train_') and
               e['vc'].startswith('train_') for e in examples[sp])
assert all(e['hub_domain'] == 'hub_valid' and e['vp'].startswith('valid_') and
           e['vc'].startswith('valid_') for e in examples['test'])
patients = {sp: {e['patient'] for e in examples[sp]} for sp in examples}
assert patients['train'].isdisjoint(patients['tune'])
assert patients['train'].isdisjoint(patients['test'])
assert patients['tune'].isdisjoint(patients['test'])
assert examples['train'], 'No usable train_* examples: finish/cache more Hub training volumes.'
assert examples['tune'], 'No usable tune examples: increase data or TUNE_FRAC.'
assert examples['test'], 'No usable valid_* examples for final test.'
if REQUIRE_COMPLETE_HUB_VALID_FEATURES:
    assert not missing_hub_valid, (
        f'{len(missing_hub_valid)} labeled Hub-validation pairs lack one/both cached features; '
        f'finish encode_split("valid") before evaluation. First missing: {missing_hub_valid[:3]}')

print('candidate labeled pairs:', dict(candidate_pairs))
print('usable cached pairs    :', dict(usable_pairs))
for sp in ['train', 'tune', 'test']:
    cc = Counter(e['y'] for e in examples[sp])
    print(f'{sp:5}: {len(examples[sp]):5} ex / {len(patients[sp]):4} patients  '
          f'(worsened={cc[0]} stable={cc[1]} improved={cc[2]})')
print('skipped:', dict(skipped))
print('LEAKAGE CHECK PASSED: optimizer=train_* only; tune=train_* only; final test=valid_* only')
FINDINGS = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']
assert {e['finding'] for sp in examples for e in examples[sp]} <= set(FINDINGS)
print('canonical findings:', len(FINDINGS))

## 6. Build finding×class text prototypes with the CT-CLIP text tower (cached)
For each finding f: average a small paraphrase bank per class → (3, 512) unit prototypes.

In [ ]:
TEMPLATES = {
    'worsened': ['{f} has increased compared to the prior study',
                 '{f} has worsened since the previous exam',
                 'interval enlargement of {f}', 'new {f}', 'increased {f}'],
    'stable':   ['{f} is unchanged compared to the prior study',
                 'stable {f} with no interval change',
                 'no significant change in {f}', '{f} appears similar to prior'],
    'improved': ['{f} has decreased compared to the prior study',
                 '{f} has improved since the previous exam',
                 'interval decrease of {f}', '{f} has resolved', 'decreased {f}'],
}

def l2np(x): return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

if os.path.exists(PROTO_PT):
    PROTO = torch.load(PROTO_PT); print('loaded cached prototypes for', len(PROTO), 'findings')
else:
    PROTO = {}
    from tqdm.auto import tqdm
    for f in tqdm(FINDINGS):
        rows = []
        for c in CLASSES:
            prompts = [t.format(f=f.lower()) for t in TEMPLATES[c]]
            e = emb.embed_texts(prompts, normalize=True).numpy()   # (k,512)
            rows.append(l2np(e.mean(0)))
        PROTO[f] = torch.tensor(np.stack(rows)).float()            # (3,512)
    torch.save(PROTO, PROTO_PT)
    print('built + cached prototypes for', len(PROTO), 'findings ->', PROTO_PT)

## 7. Tensorize splits (+ optional evidence embeddings for contrastive aux)

In [ ]:
USE_CONTRASTIVE    = True       # add InfoNCE(d, contrastive-target text) aux loss
CONTRASTIVE_TARGET = 'dynamic'  # 'dynamic' = per-pair comparison sentences (recommended); 'evidence' = legacy per-finding quote

def embed_target(exs, field=CONTRASTIVE_TARGET):
    uniq = sorted({e[field] for e in exs if e.get(field, '').strip()})
    cache = {}
    if uniq:
        B = 64
        vecs = []
        for i in range(0, len(uniq), B):
            vecs.append(emb.embed_texts(uniq[i:i+B], normalize=True))
        vecs = torch.cat(vecs, 0)
        cache = {s: vecs[j] for j, s in enumerate(uniq)}
    zero = torch.zeros(512)
    return torch.stack([cache.get(e.get(field, ''), zero) for e in exs])

def tensorize(exs):
    VP = torch.stack([POOLED[e['vp']] for e in exs])
    VC = torch.stack([POOLED[e['vc']] for e in exs])
    PR = torch.stack([PROTO[e['finding']] for e in exs])   # (N,3,512)
    Y  = torch.tensor([e['y'] for e in exs], dtype=torch.long)
    Fn = [e['finding'] for e in exs]
    PID = torch.tensor([e['pid'] for e in exs], dtype=torch.long)
    EV = embed_target(exs) if USE_CONTRASTIVE else None
    return VP, VC, PR, Y, Fn, PID, EV

DATA = {sp: tensorize(examples[sp]) for sp in ['train', 'tune', 'test']}
cnt = Counter(DATA['train'][3].tolist()); tot = sum(cnt.values())
W = torch.tensor([tot / (3 * max(cnt[i], 1)) for i in range(3)], dtype=torch.float32)
print('class weights (w/s/i):', [round(x, 3) for x in W.tolist()])

## 8. The trainable difference module (the ONLY trained component)

In [ ]:
class DifferenceTransformer(nn.Module):
    def __init__(self, d_in=512, d_model=256, n_layers=2, n_heads=4, dropout=0.1,
                 antisym=False, magnitude=False):
        super().__init__()
        self.W = nn.Linear(d_in, d_model)
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)      # prior / current
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02)    # learnable query
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4,
                                           dropout=dropout, batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, d_in)                          # back to 512-d joint space
        self.mag_head = nn.Linear(d_model, 1) if magnitude else None
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07)))
        self.antisym = antisym

    def _pass(self, vp, vc):
        B = vp.size(0)
        tp = self.W(vp) + self.role[0]
        tc = self.W(vc) + self.role[1]
        ed = self.e_diff.expand(B, -1)
        h = self.enc(torch.stack([ed, tp, tc], dim=1))                # (B,3,d_model)
        hdiff = h[:, 0]
        mag = self.mag_head(hdiff).squeeze(-1) if self.mag_head is not None else None
        return self.head(hdiff), mag                                  # (B,512), (B,)|None

    def forward(self, vp, vc):
        vd, mag = self._pass(vp, vc)
        if self.antisym:
            vd_rev, _ = self._pass(vc, vp)
            vd = vd - vd_rev                                          # stable -> 0 by construction
        return vd, mag

def logits_from(vd, proto, logit_scale):
    vd = F.normalize(vd, dim=-1)
    pr = F.normalize(proto, dim=-1)
    cos = torch.einsum('bd,bkd->bk', vd, pr)                          # (B,3)
    return logit_scale.exp().clamp(max=100) * cos

npar = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print('module params:', f'{npar(DifferenceTransformer()):,}')

## 9. Train on Hub `train_*` only (early-stop on patient-held-out `train_*` tune)

In [ ]:
# ---- ablation knobs ----
ANTISYM   = False    # d = g(c,p) - g(p,c)  (stable == 0 vector)
MAGNITUDE = False    # add ||change|| head + BCE(stable vs change); routes stable
D_MODEL, EPOCHS, LR, BATCH, PATIENCE = 256, 120, 1e-3, 256, 20
LAMBDA_CON, LAMBDA_MAG = 0.5, 0.5

model = DifferenceTransformer(d_model=D_MODEL, antisym=ANTISYM, magnitude=MAGNITUDE).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
ce = nn.CrossEntropyLoss(weight=W.to(DEVICE))

def run(split, train=False):
    VP, VC, PR, Y, Fn, PID, EV = DATA[split]
    idxs = torch.randperm(len(Y)) if train else torch.arange(len(Y))
    model.train() if train else model.eval()
    all_y, all_p, tot = [], [], 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for i in range(0, len(idxs), BATCH):
            b = idxs[i:i+BATCH]
            vp, vc, pr, y = VP[b].to(DEVICE), VC[b].to(DEVICE), PR[b].to(DEVICE), Y[b].to(DEVICE)
            vd, mag = model(vp, vc)
            lg = logits_from(vd, pr, model.logit_scale)
            loss = ce(lg, y)
            if MAGNITUDE and mag is not None:
                is_change = (y != C2I['stable']).float()
                loss = loss + LAMBDA_MAG * F.binary_cross_entropy_with_logits(mag, is_change)
            if USE_CONTRASTIVE and EV is not None:
                ev = EV[b].to(DEVICE)
                pid = PID[b]
                # dynamic sentences are per-PAIR: keep ONE row per unique pair (non-empty text)
                sel, seen = [], set()
                for j in range(len(b)):
                    if ev[j].norm() <= 0: continue
                    p = int(pid[j])
                    if p in seen: continue
                    seen.add(p); sel.append(j)
                if len(sel) > 1:
                    sel = torch.tensor(sel, device=DEVICE)
                    zc = F.normalize(vd[sel], dim=-1) @ F.normalize(ev[sel], dim=-1).t()
                    zc = model.logit_scale.exp().clamp(max=100) * zc
                    tgt = torch.arange(len(sel), device=DEVICE)
                    loss = loss + LAMBDA_CON * 0.5 * (F.cross_entropy(zc, tgt) + F.cross_entropy(zc.t(), tgt))
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
            all_y += y.cpu().tolist(); all_p += lg.argmax(1).cpu().tolist()
    mf1 = f1_score(all_y, all_p, labels=[0,1,2], average='macro', zero_division=0)
    return tot/len(idxs), mf1, np.array(all_y), np.array(all_p)

best, best_state, bad = -1, None, 0
for ep in range(1, EPOCHS+1):
    tl, _, _, _ = run('train', train=True)
    _, vf1, _, _ = run('tune')
    if vf1 > best:
        best, best_state, bad = vf1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
    else:
        bad += 1
    if ep % 10 == 0 or ep == 1:
        print(f'ep {ep:3}  loss {tl:.3f}  tune_macroF1 {vf1:.3f}  (best {best:.3f})')
    if bad >= PATIENCE:
        print(f'early stop @ ep {ep}  best tune macro-F1 {best:.3f}'); break
model.load_state_dict(best_state)
print('restored best. train-domain tune macro-F1 =', round(best, 3),
      f'| ANTISYM={ANTISYM} MAGNITUDE={MAGNITUDE} CONTRASTIVE={USE_CONTRASTIVE}')

## 10. FINAL TEST — CT-RATE Hub `valid_*` only

Run once after architecture/hyperparameters are fixed. Do not tune from these numbers.

In [ ]:
_, macro, y, pred = run('test')
percls = f1_score(y, pred, labels=[0,1,2], average=None, zero_division=0)
acc = (y == pred).mean()
maj = Counter(DATA['train'][3].tolist()).most_common(1)[0][0]
maj_macro = f1_score(y, np.full_like(y, maj), labels=[0,1,2], average='macro', zero_division=0)
assert all(e['hub_domain'] == 'hub_valid' for e in examples['test'])
print('=== FINAL HUB-VALIDATION TEST — per class ===')
print(f'accuracy : {acc:.3f}')
print(f'macro-F1 : {macro:.3f}   (always-{CLASSES[maj]} ref = {maj_macro:.3f})')
for i, c in enumerate(CLASSES):
    print(f'  F1 {c:9}: {percls[i]:.3f}')
print('\nconfusion (rows=true, cols=pred; order w/s/i):')
print(confusion_matrix(y, pred, labels=[0,1,2]))

## 11. FINAL HUB-VALIDATION TEST — per disease (finding)

In [ ]:
_, _, _, _ = run('test')  # ensure eval mode
VP, VC, PR, Y, Fn, _, _ = DATA['test']
model.eval()
with torch.no_grad():
    vd, _ = model(VP.to(DEVICE), VC.to(DEVICE))
    pred = logits_from(vd, PR.to(DEVICE), model.logit_scale).argmax(1).cpu().numpy()
y = Y.numpy()
by_f = defaultdict(lambda: {'y': [], 'p': []})
for yi, pi, fi in zip(y, pred, Fn):
    by_f[fi]['y'].append(yi); by_f[fi]['p'].append(pi)
rows = []
for f, d in by_f.items():
    yy, pp = np.array(d['y']), np.array(d['p'])
    present = sorted(set(yy.tolist()))
    rows.append((f, len(yy), (yy==pp).mean(), f1_score(yy, pp, labels=present, average='macro', zero_division=0)))
rows.sort(key=lambda r: -r[1])
print('=== FINAL HUB-VALIDATION TEST — per disease ===')
print(f"{'finding':<34}{'n':>5}{'acc':>7}{'macroF1*':>10}")
for f, n, a, mf1 in rows[:25]:
    print(f'{f:<34}{n:>5}{a:>7.3f}{mf1:>10.3f}')
print('\n* macro-F1 over classes actually present for that finding')

## Notes / ablations
- **Domain split is authoritative:** optimization and early stopping use only CT-RATE `train_*`;
  final evaluation uses only `valid_*`. The old `subset_pairs.csv` split is ignored. Partial
  train caching merely reduces usable training pairs; it never promotes validation examples.
- **Do not tune on final test:** all architecture/ablation choices must use `tune` (a deterministic
  patient-held-out subset of Hub train). Inspect Hub validation only after choices are frozen.
- **Only the module in cell 8 trains** (~few M params); CT-CLIP encoders + text prototypes are frozen.
- **Ablations** (cell 9): `ANTISYM=True` (antisymmetric readout → stable=0 vector), `MAGNITUDE=True`
  (adds a change-vs-stable head), `USE_CONTRASTIVE=True` in cell 7 (InfoNCE aligning `d` with the
  per-pair `dynamic_sentences` — the difference-embedding ↔ report-text idea).
- **Baselines to beat:** always-`stable` (printed), and `v_current − v_prior` fed to the same head.
- **Compare to MERLIN v1** (`train_v1_ctrate.ipynb`): same protocol, chest-native CT-CLIP features —
  does the domain-matched encoder + difference module lift worsened/improved F1?
- **Contrastive target = per-pair `dynamic_sentences`** (from `medgemma_labels (2).jsonl`), set via
  `CONTRASTIVE_TARGET='dynamic'` in cell 7. These are the LLM-extracted comparison sentences, so
  InfoNCE aligns `d` with real change language (not the noisy per-finding `evidence`). The loss is
  de-duplicated to one row per unique pair per batch (dynamic text is per-pair, not per-finding).
  To add the full two-path objective, also align `v_current` with `static_sentences`.